# Build an LLM from Scratch
## Coding Attention Mechanisms
### Attending to Different Parts of the Input with Self-Attention
#### Simple Self-Attention Mechanism Without Trainable Weights

In [1]:
import torch

In [2]:
# Embedded text
inputs = torch.tensor(
    [[0.43, 0.15, 0.89],
     [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64],
     [0.22, 0.58, 0.33],
     [0.77, 0.25, 0.10],
     [0.05, 0.80, 0.55]]
)
inputs = inputs
query = inputs[1]
print(f"Input query: {query}")

Input query: tensor([0.5500, 0.8700, 0.6600])


In [3]:
attn_scores = inputs @ query.T
print(f"Attention scores: {attn_scores}")

Attention scores: tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


/tmp/nix-shell.rfWuzK/ipykernel_46215/2785625054.py:1: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4320.)
  attn_scores = inputs @ query.T


This is getting the dot product for the given input for all inputs. It's the same as if iterating over each input and then getting the dot product:

In [4]:
for i, x_i in enumerate(inputs):
    print(torch.dot(x_i, query))

tensor(0.9544)
tensor(1.4950)
tensor(1.4754)
tensor(0.8434)
tensor(0.7070)
tensor(1.0865)


Now the attention scores are normalised. This is done so they become a probability distribution over tokens.

In [5]:
attn_scores_weighted = attn_scores / attn_scores.sum()
print(f"Attention scores weighted: {attn_scores_weighted}")
print(f"Sum of scores: {attn_scores_weighted.sum():.2f}")

Attention scores weighted: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum of scores: 1.00


In practice, softmax does a better job. All values will sit between 0 and 1 and high scores now get a larger, whilst small ones get smaller.

In [6]:
def softmax_naive(x):
    """Don't use this as it's numerically unstable."""
    return torch.exp(x) / torch.exp(x).sum()

In [7]:
softmax_scores = torch.softmax(attn_scores, dim=0)
print(f"Softmax scores: {softmax_scores}")

Softmax scores: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


In [8]:
context_vector = softmax_scores @ inputs
print(f"Context vector: {context_vector}")

Context vector: tensor([0.4419, 0.6515, 0.5683])


The softmax scores say how related each key is to the query. The context vector is a new representation for the query token, blending all the input embeddings by those scores. In effect, the given query has added information of all the other inputs.

In [9]:
print(f"Query: {query} was transformed into a context vector: {context_vector}")

Query: tensor([0.5500, 0.8700, 0.6600]) was transformed into a context vector: tensor([0.4419, 0.6515, 0.5683])


In [10]:
full_attn_scores = inputs @ inputs.T
print("Full attention scores:")
print(full_attn_scores)

Full attention scores:
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


Notice how the second row is the same as attention score from before for `inputs[1]`:

In [11]:
attn_scores

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

So each row is the attention scores for a given query to the other keys.

In [12]:
full_attn_weights = torch.softmax(full_attn_scores, dim=1)

print("Full attention weights:")
full_attn_weights

Full attention weights:


tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [13]:
full_context_vecs = full_attn_weights @ inputs

print("Full context vectors:")
full_context_vecs

Full context vectors:


tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

Each row represents a given query's added context from all other keys/tokens before and after it.

### Inplementing Self-Attention with Trainable Weights
#### Computing the Attention Weights Step by Step

In [14]:
d_in = inputs.shape[1]
d_out = 3

torch.manual_seed(123)
# Parameter makes this trainable
W_query = torch.nn.Parameter(torch.rand(d_in, d_out))
W_key = torch.nn.Parameter(torch.rand(d_in, d_out))
W_value = torch.nn.Parameter(torch.rand(d_in, d_out))

print("Query matrix:")
print(W_query)
print("===============\n")

print("Key matrix:")
print(W_key)
print("===============\n")


print("Value matrix:")
print(W_value)
print("===============\n")

Query matrix:
Parameter containing:
tensor([[0.2961, 0.5166, 0.2517],
        [0.6886, 0.0740, 0.8665],
        [0.1366, 0.1025, 0.1841]], requires_grad=True)

Key matrix:
Parameter containing:
tensor([[0.7264, 0.3153, 0.6871],
        [0.0756, 0.1966, 0.3164],
        [0.4017, 0.1186, 0.8274]], requires_grad=True)

Value matrix:
Parameter containing:
tensor([[0.3821, 0.6605, 0.8536],
        [0.5932, 0.6367, 0.9826],
        [0.2745, 0.6584, 0.2775]], requires_grad=True)



In [15]:
keys = inputs @ W_key
queries = inputs @ W_query
values = inputs @ W_value

print(f"Keys shape: {keys.shape}")
print(f"Queries shape: {queries.shape}")
print(f"Values shape: {values.shape}")

Keys shape: torch.Size([6, 3])
Queries shape: torch.Size([6, 3])
Values shape: torch.Size([6, 3])


Let's calculate the context vector for a single query first:

In [16]:
d_k = keys.shape[1]

attn_scores = queries[1] @ keys.T 
softmax_scores = torch.softmax(attn_scores / (d_k ** 0.5), dim=0)
context_vector = softmax_scores.T @ values

context_vector

tensor([0.6864, 1.0577, 1.1389], grad_fn=<SqueezeBackward4>)

Now for the whole context:

In [17]:
attn_scores = queries @ keys.T
softmax_scores = torch.softmax(attn_scores / (d_k ** 0.5), dim=1)
context_vecs = softmax_scores @ values

context_vecs

tensor([[0.6692, 1.0276, 1.1106],
        [0.6864, 1.0577, 1.1389],
        [0.6860, 1.0570, 1.1383],
        [0.6738, 1.0361, 1.1180],
        [0.6711, 1.0307, 1.1139],
        [0.6783, 1.0441, 1.1252]], grad_fn=<MmBackward0>)

### Implementing a Compact Self-Attention Class

In [18]:
# torch.nn.Module is the base class for anything with learnable weights
class SelfAttentionV1(torch.nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = torch.nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = torch.nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = torch.nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        d_k = keys.shape[-1]

        attn_scores = queries @ keys.T
        softmax_scores = torch.softmax(attn_scores / (d_k ** 0.5), dim=-1)
        return softmax_scores @ values

In [19]:
torch.manual_seed(123)
sa = SelfAttentionV1(d_in=3, d_out=3)
sa(inputs)

tensor([[0.6692, 1.0276, 1.1106],
        [0.6864, 1.0577, 1.1389],
        [0.6860, 1.0570, 1.1383],
        [0.6738, 1.0361, 1.1180],
        [0.6711, 1.0307, 1.1139],
        [0.6783, 1.0441, 1.1252]], grad_fn=<MmBackward0>)

You can see this gives the same result as the manual effort above.

There is a better way of doing this using `torch.nn.Linear`.

In [20]:
class SelfAttentionV2(torch.nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in, d_out, bias=False)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=False)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=False)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        d_k = keys.shape[-1]

        attn_scores = queries @ keys.T
        softmax_scores = torch.softmax(attn_scores / (d_k ** 0.5), dim=-1)
        return softmax_scores @ values

In [21]:
torch.manual_seed(123)
sa = SelfAttentionV2(d_in=3, d_out=3)
sa(inputs)

tensor([[ 0.2633,  0.4277, -0.1353],
        [ 0.2641,  0.4296, -0.1350],
        [ 0.2641,  0.4296, -0.1350],
        [ 0.2647,  0.4316, -0.1381],
        [ 0.2642,  0.4303, -0.1373],
        [ 0.2648,  0.4316, -0.1375]], grad_fn=<MmBackward0>)

Note that the values are not the same as before, this is because the K/Q/V weights are initialised differently. It's more standard to not use `torch.nn.Parameter`.

## Hiding Future Words with Casual Attention
### Applying a Causual Attention Mask

Currently we have this for the attention pattern:

In [22]:
queries = sa.W_query(inputs)
keys = sa.W_key(inputs)
values = sa.W_value(inputs)

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / d_k ** 0.5, dim=1)
attn_weights

tensor([[0.1466, 0.1642, 0.1642, 0.1784, 0.1686, 0.1780],
        [0.1365, 0.1743, 0.1743, 0.1727, 0.1685, 0.1737],
        [0.1368, 0.1742, 0.1741, 0.1728, 0.1685, 0.1737],
        [0.1494, 0.1725, 0.1725, 0.1686, 0.1677, 0.1694],
        [0.1497, 0.1691, 0.1690, 0.1720, 0.1680, 0.1723],
        [0.1456, 0.1743, 0.1743, 0.1685, 0.1678, 0.1694]],
       grad_fn=<SoftmaxBackward0>)

But actually, we don't want future words to affect a given query. These are masked out:

In [23]:
attn_pattern = torch.tril(attn_weights)
attn_pattern

tensor([[0.1466, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1365, 0.1743, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1368, 0.1742, 0.1741, 0.0000, 0.0000, 0.0000],
        [0.1494, 0.1725, 0.1725, 0.1686, 0.0000, 0.0000],
        [0.1497, 0.1691, 0.1690, 0.1720, 0.1680, 0.0000],
        [0.1456, 0.1743, 0.1743, 0.1685, 0.1678, 0.1694]],
       grad_fn=<TrilBackward0>)

This is short hand for multiplying a triangle matrix (1s on the bottom, 0s on top) by the attention weights. Now the rows no longer add up to 1. This is address like this:

In [24]:
attn_pattern / attn_pattern.sum(dim=1, keepdim=True)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4392, 0.5608, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2820, 0.3591, 0.3589, 0.0000, 0.0000, 0.0000],
        [0.2253, 0.2602, 0.2601, 0.2544, 0.0000, 0.0000],
        [0.1809, 0.2043, 0.2042, 0.2078, 0.2029, 0.0000],
        [0.1456, 0.1743, 0.1743, 0.1685, 0.1678, 0.1694]],
       grad_fn=<DivBackward0>)

To avoid having do re-normalise the attention scores, a mask of `-inf` can be fed into the softmax to keep the upper triangle 0.

In [25]:
triu_mask = torch.triu(torch.ones(queries.shape[0], queries.shape[0], dtype=torch.bool), diagonal=1)
masked_attn_scores = attn_scores.masked_fill(triu_mask, -torch.inf)
masked_attn_scores

tensor([[-0.4028,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.2623,  0.1610,    -inf,    -inf,    -inf,    -inf],
        [-0.2630,  0.1553,  0.1546,    -inf,    -inf,    -inf],
        [-0.0989,  0.1501,  0.1497,  0.1111,    -inf,    -inf],
        [-0.2004,  0.0102,  0.0098,  0.0397, -0.0013,    -inf],
        [-0.1048,  0.2070,  0.2065,  0.1480,  0.1407,  0.1575]],
       grad_fn=<MaskedFillBackward0>)

In [26]:
torch.softmax(masked_attn_scores / d_k ** 0.5, dim=1)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4392, 0.5608, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2820, 0.3591, 0.3589, 0.0000, 0.0000, 0.0000],
        [0.2253, 0.2602, 0.2601, 0.2544, 0.0000, 0.0000],
        [0.1809, 0.2043, 0.2042, 0.2078, 0.2029, 0.0000],
        [0.1456, 0.1743, 0.1743, 0.1685, 0.1678, 0.1694]],
       grad_fn=<SoftmaxBackward0>)

As you can see, it's the same as before, except there is one fewer step.

### Masking Additional Attention Weights with Dropout

The dropout mask will pick random positions in the attention pattern and mask them out. This is to aid in training by reducing overfitting. It will rely less on certain positions.

In [27]:
torch.manual_seed(123)
dropout_layer = torch.nn.Dropout(0.5)

In [28]:
dropout_layer(torch.ones(6, 6))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])

To maintain the same average per row, dropout is adding numbers to rescale.

### Implementing a Compact Causal Self-Attention Class

In [29]:
class CausalAttention(torch.nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in, d_out, bias=False)
        self.W_key = torch.nn.Linear(d_in, d_out, bias=False)
        self.W_value = torch.nn.Linear(d_in, d_out, bias=False)
        self.dropout = torch.nn.Dropout(dropout)
        # Because arbitrary matrices won't transfer over to the GPU, this is buffer is needed
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1),
        )

    def forward(self, x):
        batch, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        d_k = keys.shape[-1]

        attn_scores = queries @ keys.transpose(1, 2)
        # _ ops are in-place, it avoids unnecessary copies
        attn_scores.masked_fill_(self.mask[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / d_k ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        return attn_weights @ values

`transpose(1, 2)` transposes the inner weights for each batch.

```
axis 0 = batch       (= 0)
axis 1 = keys x-axis (= 1)
axis 2 = keys y-axis (= 2)
```

In [30]:
torch.manual_seed(789)

batch = torch.stack((inputs, inputs))
ca = CausalAttention(
    d_in=inputs.shape[1],
    d_out=inputs.shape[1],
    context_length=6,
    dropout=0,
)

In [31]:
print("Each batch's context vectors:")
ca.forward(batch)

Each batch's context vectors:


tensor([[[ 0.3253, -0.5116, -0.1020],
         [ 0.4499, -0.5958, -0.0050],
         [ 0.4909, -0.6204,  0.0269],
         [ 0.4473, -0.5584,  0.0417],
         [ 0.4247, -0.4955,  0.0352],
         [ 0.4166, -0.4996,  0.0483]],

        [[ 0.3253, -0.5116, -0.1020],
         [ 0.4499, -0.5958, -0.0050],
         [ 0.4909, -0.6204,  0.0269],
         [ 0.4473, -0.5584,  0.0417],
         [ 0.4247, -0.4955,  0.0352],
         [ 0.4166, -0.4996,  0.0483]]], grad_fn=<UnsafeViewBackward0>)

### Extending Single-Head Attention to Multi-Head Attention
#### Stacking Multiple Single-Head Attention Layers

With multi-head attention, the context vectors from each head are concatenated on top of each other.

In [32]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads):
        super().__init__()
        self.heads = torch.nn.ModuleList([
            CausalAttention(
                d_in=d_in,
                d_out=d_out,
                context_length=context_length,
                dropout=dropout,
            )
            for _ in range(num_heads)
        ])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [33]:
torch.manual_seed(123)

mha = MultiHeadAttention(
    d_in=3,
    d_out=3,
    context_length=batch.shape[1],
    dropout=0,
    num_heads=2,
)
mha(batch)

tensor([[[ 0.3326,  0.5659, -0.3132,  0.0752,  0.4566,  0.2729],
         [ 0.3456,  0.5650, -0.2237,  0.0313,  0.5977,  0.3053],
         [ 0.3440,  0.5604, -0.2000,  0.0178,  0.6413,  0.3138],
         [ 0.3103,  0.4941, -0.1606,  0.0089,  0.5729,  0.2785],
         [ 0.2430,  0.4287, -0.1643,  0.0071,  0.5566,  0.2514],
         [ 0.2648,  0.4316, -0.1375,  0.0023,  0.5363,  0.2508]],

        [[ 0.3326,  0.5659, -0.3132,  0.0752,  0.4566,  0.2729],
         [ 0.3456,  0.5650, -0.2237,  0.0313,  0.5977,  0.3053],
         [ 0.3440,  0.5604, -0.2000,  0.0178,  0.6413,  0.3138],
         [ 0.3103,  0.4941, -0.1606,  0.0089,  0.5729,  0.2785],
         [ 0.2430,  0.4287, -0.1643,  0.0071,  0.5566,  0.2514],
         [ 0.2648,  0.4316, -0.1375,  0.0023,  0.5363,  0.2508]]],
       grad_fn=<CatBackward0>)

Note that the output of this is now `6`, where as before it was `3`. This is because each context vector is now concatenated. If there were `3` heads, then the output would be `9`.

### Inplimenting Multi-Head Attention with Weight Spilts

There is a more efficient way that this can be implimented. The problem is that each head is sequentially called. They can be executed in parallel.

In [35]:
keys = sa.W_key(batch)
queries = sa.W_query(batch)
values = sa.W_value(batch)

batches = batch.shape[0]
num_tokens = batch.shape[1]
num_heads = 3
head_dim = batch.shape[-1] // num_heads

# Implicitly split the matrix by adding a `num_heads` dimension
# Unroll the last dim: (batch, num_tokens, d_out) -> (batch, num_tokens, num_heads, head_dim)
keys = keys.view(batches, num_tokens, num_heads, head_dim)
queries = queries.view(batches, num_tokens, num_heads, head_dim)
values = values.view(batches, num_tokens, num_heads, head_dim)

# Transpose: (batch, num_tokens, num_heads, head_dim) -> (batch, num_heads, num_tokens, head_dim)
keys = keys.transpose(1, 2)
queries = queries.transpose(1, 2)
values = values.transpose(1, 2)

# Compute scaled dot-product attention
attn_scores = queries @ keys.transpose(2, 3)